# vertical-shmup on Colab（PPO 訓練）

Runtime -> Change runtime type -> T4 GPU，然後依序執行每一格。

程式碼會自動從 GitHub（`rayliangg/vertical-shmup`）下載，不用另外上傳。

Colab 停止時會刪掉所有檔案（關掉分頁或閒置太久就會停）。長時間訓練請設 `USE_DRIVE = True`，
模型、紀錄、報告都會存在 Google Drive，**重新執行訓練那一格就會從上次的地方接著跑**。

所有套件都是 2022 年 5 月 27 日以前的版本：Python 3.10.4、PyTorch 1.11.0（CUDA 11.3）、Stable Baselines3 1.5.0、gym 0.21.0。

In [ ]:
# 訓練設定
import os
EPISODES = 5000                       # 玩滿幾回合就停
SAVE_EPISODES = "100,500,2000,5000"   # 這幾個回合各存一份模型（網頁上的 agent、報告用）
DEVICE = "cuda"                       # cuda = GPU，cpu = 不用 GPU
N_ENVS = os.cpu_count()               # 同時玩幾局（每局一個程序）
USE_DRIVE = True                      # True = 結果存到 Google Drive（MyDrive/vertical-shmup）
REPO_URL = "https://github.com/rayliangg/vertical-shmup"   # 空白 = 改用 Drive 上的 vertical-shmup.zip
BRANCH = "main"

In [ ]:
# 結果存在哪裡
import os
from google.colab import drive
if USE_DRIVE or not REPO_URL:
    drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/vertical-shmup" if USE_DRIVE else "/content/results"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

In [ ]:
# 安裝：程式碼 + Python 3.10.4 + 2022 年 5 月以前的套件（約 3–5 分鐘，log 在 /tmp/pip.log）
os.chdir("/content")
if REPO_URL:
    if not os.path.exists("vertical-shmup"):
        !git clone -q -b {BRANCH} {REPO_URL} vertical-shmup
    !cd vertical-shmup && git pull -q
else:
    !rm -rf vertical-shmup && unzip -q -o /content/drive/MyDrive/vertical-shmup.zip -d /content
os.chdir("/content/vertical-shmup")

# Colab 內建 Python 3.12 裝不了 2022 年 5 月以前的套件，所以另外裝 Python 3.10.4
PY = "/content/py310/bin/python"
if not os.path.exists(PY):
    !wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
    !/content/bin/micromamba create -y -q -p /content/py310 -c conda-forge python=3.10.4 pip=22.1.1 setuptools=62.3.2 wheel=0.37.1 > /tmp/pip.log 2>&1
# gym 0.21 要用 2022 年的 setuptools 62.3 建置，所以先單獨裝；torch 用 CUDA 11.3 的版本
!{PY} -m pip install -q --no-build-isolation gym==0.21.0 >> /tmp/pip.log 2>&1 && {PY} -m pip install -q -r requirements.txt "torch==1.11.0+cu113" --extra-index-url https://download.pytorch.org/whl/cu113 >> /tmp/pip.log 2>&1 && {PY} -m pip install -q --no-deps -e . >> /tmp/pip.log 2>&1 && echo "install ok" || tail -20 /tmp/pip.log

In [ ]:
# 檢查 GPU 和套件版本
!nvidia-smi -L
!{PY} -c "import platform, torch, stable_baselines3 as sb3, gym, numpy; print('python', platform.python_version(), '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| sb3', sb3.__version__, '| gym', gym.__version__, '| numpy', numpy.__version__)"

In [ ]:
# 快速測試：玩 3 回合（約 1 分鐘），確定安裝沒問題
!{PY} -m shmup train --device {DEVICE} --n-envs 2 --max-episodes 3 --save-episodes 3 --eval-every 100000000 --milestone-eval 1 --out-dir /tmp/quick_test 2>&1 | grep -E "PPO|回合|共 "

In [ ]:
# 訓練（結果每個里程碑都會存；斷線後重新執行第 3、4 格和這一格就會接著跑）
!{PY} -u -m shmup train --device {DEVICE} --n-envs {N_ENVS} --max-episodes {EPISODES} --save-episodes {SAVE_EPISODES} --out-dir {OUT}/ppo --resume auto 2>&1 | tee -a {OUT}/train.log

In [ ]:
# 結果表 + 學習曲線
from IPython.display import Image, Markdown, display
!{PY} -m shmup report --run-dir {OUT}/ppo
text = open(OUT + "/ppo/report/results.md").read()
display(Markdown(text.replace("![學習曲線](learning_curve.png)", "")))
display(Image(OUT + "/ppo/report/learning_curve.png"))

In [ ]:
# 下載網頁版模型（USE_DRIVE = True 時也可以直接到 Google Drive 拿）
# 把 web_models.zip 裡 web/ 的檔案放到專案的 web/model/，打開 web/compare.html 就能看四個 agent 一起玩
from google.colab import files
!cd {OUT}/ppo && zip -qr {OUT}/web_models.zip web milestones.json best.json report
files.download(OUT + "/web_models.zip")